# Krishi Drishti - P09 & P10: Evaluation & TFLite Model Export
Evaluates `best.keras` on held-out test data, checks calibration, converts to TFLite (dynamic range & float16), tests top-1 consistency >=95%, and outputs MODEL_CARD.md.


In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

model = tf.keras.models.load_model("ml-training/models/best.keras")
split_df = pd.read_csv("ml-training/data/split.csv")
test_df = split_df[split_df["split"] == "test"]
with open("ml-training/data/class_names.json") as f:
    class_names = json.load(f)
print(f"Loaded model and {len(test_df)} test samples.")


### Convert to TensorFlow Lite (Dynamic Range Quantization)


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("ml-training/models/model.tflite", "wb") as f:
    f.write(tflite_model)
print(f"TFLite model exported: {len(tflite_model) / (1024*1024):.2f} MB")


### Top-1 Agreement Check between Keras and TFLite


In [ ]:
interpreter = tf.lite.Interpreter(model_path="ml-training/models/model.tflite")
interpreter.allocate_tensors()
in_idx = interpreter.get_input_details()[0]["index"]
out_idx = interpreter.get_output_details()[0]["index"]

sample_tests = test_df.sample(20, random_state=42)
matches = 0
for _, row in sample_tests.iterrows():
    img = Image.open(row["path"]).convert("RGB").resize((224, 224))
    arr = np.expand_dims(np.array(img, dtype=np.float32), 0)
    k_pred = int(np.argmax(model(arr, training=False)))
    interpreter.set_tensor(in_idx, arr)
    interpreter.invoke()
    t_pred = int(np.argmax(interpreter.get_tensor(out_idx)))
    if k_pred == t_pred:
        matches += 1

print(f"Top-1 Agreement: {matches / 20 * 100:.1f}% (Required >= 95%)")
assert (matches / 20) >= 0.95
